In [ ]:
!pip install wfdb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 28.2 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.1 which is incompatible.


In [ ]:
import wfdb
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from google.colab import files

In [ ]:
# - slices every SVA ECG record into windows
# - labels windows bassed on annotations
# - stores segments & labels for later training

# 0 = no SVA
# 1 = SVA
# 2 = pre-SVA

all_windows = [] # will store 5-sec snippets (windows) of ECG data
                 # contains 640 numbers per snippet (5 sec * 128 samples / sec) -> IS A MATRIX
                 # ea/ num represents the electrical activity of the heart
                 # window could look like:
                 # [0.245, 0.251, 0.248, 0.243, ..., 0.256] w/ 640 #s total
                 # these are the raw voltage readings that make squiggly ECG lines on plotter

all_labels = [] # contains the classifications for ea/ window
                # corresponds to one window in all_windows
                # single number: 0, 1, 2

fs = 128 # default ECG sampling rate of 128 Hz
window_duration = 12 # seconds per window
overlap_duration = (window_duration - 0.5) # seconds per window overlap
window_size = int(window_duration*fs) # numbers in ea/ window
step_size = int((window_duration - overlap_duration) * fs) # use increment thru ecg data

svc_record_names = wfdb.get_record_list('svdb') # gets list of records in svdb

sva_symbols = ['S', 'J', 'a'] # specifies which annotation symbols indicate SVA
noise_symbols = ['Q', '~', '|'] # consider adding V, F, B, + for super clean training data

for record_name in svc_record_names: # loop thru SVA dataset, if causing error skip record
    try:
        record = wfdb.rdrecord(record_name, pn_dir='svdb') # downloads record & annotation from physionet
        annotation = wfdb.rdann(record_name, 'atr', pn_dir='svdb')

        ecg_signal = record.p_signal[:, 0] # get first EG channel
        fs = record.fs # update fs from record metadata
        window_size = int(window_duration * fs)
        step_size = int((window_duration - overlap_duration) * fs)

        ann_samples = annotation.sample # extracts annotations
        ann_symbols = annotation.symbol

        sva_event_samples = [samp for samp, sym in zip(ann_samples, ann_symbols) if sym in sva_symbols]

        for start in range(0, len(ecg_signal) - window_size + 1, step_size): # loop thru ea/ window
            end = start + window_size
            window_data = ecg_signal[start:end] # extract data

            if len(window_data) < window_size: # skip windows too short (usually @ end)
                continue

            ann_in_window = [sym for samp, sym in zip(ann_samples, ann_symbols) if start <= samp < end]

            if any(sym in noise_symbols for sym in ann_in_window):
                continue # skip weird windows

            if any(sym in sva_symbols for sym in ann_in_window):
                label = 1  # current window contains SVA beat
            else:
                # check if any sva beats occur within 5 sec after this window
                prediction_window = int(window_duration * fs)
                window_end = end
                is_presva = any(window_end < sva_time <= window_end + prediction_window for sva_time in sva_event_samples)
                label = 2 if is_presva else 0  # pre-sva or normal

            all_windows.append(window_data) # add curr window to all_windows
            all_labels.append(label)

    except Exception as e:
        print(f"skipping record {record_name} due to error: {e}")

all_windows = np.array(all_windows)
all_labels = np.array(all_labels)

print(f"total windows: {len(all_windows):,}")
print(f"window shape: {all_windows.shape}")
print(f"\nclass distributions:")
print(np.bincount(all_labels))

total windows: 231,892
window shape: (231892, 3072)

class distributions:
[114255  93798  23839]


In [ ]:
from google.colab import drive
import numpy as np
import os

drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/'
os.makedirs(save_dir, exist_ok=True)

print("saving to google drive...")
np.savez_compressed(save_dir + 'ecg_data', windows=all_windows,labels=all_labels)

print("saved to google drive!")
print(f"location: {save_dir}")

Mounted at /content/drive
saving to google drive...
saved to google drive!
location: /content/drive/MyDrive/ZiyaAhmad/


In [ ]:
# uncomment the code below if you'd like to save the ecg to your local device

# print("compressing file...")
# np.savez_compressed('ecg_data.npz', windows=all_windows, labels=all_labels, fs=fs, window_duration=window_duration,overlap_duration=overlap_duration)
# print("created ecg_data.npz (~1 GB?)")

# print("\ndownloading to computer")
# files.download('ecg_data.npz')
# print("downloaded")

compressing file...
created ecg_data.npz (~1 GB?)

downloading to computer


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

downloaded

 MOVE TO DIFF FOLDER
